%md
## Load Silver Fact Table

In [0]:
import datetime
from pyspark.sql.functions import col, count, when, lit

run_id = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
dataset_name = "silver_valuation_multiple_staging"

df = spark.table("silver_valuation_multiple_staging")
total_records = df.count()
print(f"Run ID: {run_id} | Total records: {total_records}")

Run ID: 20260818_054505 | Total records: 404


In [0]:
display(df)


deal_count,ev_bracket,median_multiple,metric_type,p25,p75,source_file,source_year,sub_vertical,valuation_id,ingestion_date
11,25m_100m_ev,1.75,ev_revenue,0.85,4.66,ma_multiples,null,advertising-agency,0,2026-08-18
14,5m_25m_ev,1.29,ev_revenue,0.85,1.67,ma_multiples,null,advertising-agency,1,2026-08-18
24,100m_500m_ev,2.38,ev_revenue,1.35,3.06,ma_multiples,null,aerospace,2,2026-08-18
13,25m_100m_ev,1.86,ev_revenue,0.76,2.93,ma_multiples,null,aerospace,3,2026-08-18
10,5m_25m_ev,0.77,ev_revenue,0.61,1.02,ma_multiples,null,aerospace,4,2026-08-18
24,over_500m_ev,14.25,ev_ebitda,12.9,16.92,ma_multiples,null,aerospace,5,2026-08-18
32,over_500m_ev,2.12,ev_revenue,1.67,3.29,ma_multiples,null,aerospace,6,2026-08-18
11,25m_100m_ev,9.5,ev_ebitda,6.8,12.2,ma_multiples,null,ambulatory-surgery-center,7,2026-08-18
13,25m_100m_ev,2.5,ev_revenue,1.95,3.2,ma_multiples,null,ambulatory-surgery-center,8,2026-08-18
20,5m_25m_ev,8.1,ev_ebitda,5.47,11.38,ma_multiples,null,ambulatory-surgery-center,9,2026-08-18


### Rules:
- Rule 1 — Null validation
- Rule 2 — Multiple validation \ positive value
- Rule 3 — Percentile consistency
- Rule 4 — Deal-count validation 

In [0]:
df_flagged = df.withColumn(
    "is_null_valid",
    when(col("source_file") == "ma_multiples",
         col("sub_vertical").isNotNull() & col("ev_bracket").isNotNull())
    .when(col("source_file") == "ma_multiples_by_year",
          col("sub_vertical").isNotNull() & col("source_year").isNotNull())
    .otherwise(lit(False))
).withColumn(
    "is_positive_valid",
    (col("median_multiple") > 0) & (col("p25") > 0) & (col("p75") > 0)
).withColumn(
    "is_percentile_valid",
    (col("p25") <= col("median_multiple")) & (col("median_multiple") <= col("p75"))
).withColumn(
    "is_deal_count_valid",
    col("deal_count") >= 10
)

display(df_flagged.select("sub_vertical", "ev_bracket", "source_year", "metric_type",
                           "is_null_valid", "is_positive_valid", "is_percentile_valid", "is_deal_count_valid").limit(10))

sub_vertical,ev_bracket,source_year,metric_type,is_null_valid,is_positive_valid,is_percentile_valid,is_deal_count_valid
advertising-agency,25m_100m_ev,null,ev_revenue,true,true,true,true
advertising-agency,5m_25m_ev,null,ev_revenue,true,true,true,true
aerospace,100m_500m_ev,null,ev_revenue,true,true,true,true
aerospace,25m_100m_ev,null,ev_revenue,true,true,true,true
aerospace,5m_25m_ev,null,ev_revenue,true,true,true,true
aerospace,over_500m_ev,null,ev_ebitda,true,true,true,true
aerospace,over_500m_ev,null,ev_revenue,true,true,true,true
ambulatory-surgery-center,25m_100m_ev,null,ev_ebitda,true,true,true,true
ambulatory-surgery-center,25m_100m_ev,null,ev_revenue,true,true,true,true
ambulatory-surgery-center,5m_25m_ev,null,ev_ebitda,true,true,true,true


- Rule 5 — Duplicate validation 

In [0]:
from pyspark.sql.window import Window

business_key = ["sub_vertical", "ev_bracket", "metric_type", "source_year"]
window_key = Window.partitionBy(*business_key)

df_flagged = df_flagged.withColumn("key_count", count("*").over(window_key))
df_flagged = df_flagged.withColumn("is_unique_valid", col("key_count") == 1)

duplicate_count = df_flagged.filter(col("is_unique_valid") == False).count()
print(f"Duplicate records found: {duplicate_count}")

Duplicate records found: 0


- Rule 6 — Range validation / outlier detection

In [0]:
df_flagged = df_flagged.withColumn(
    "is_range_valid",
    when(col("metric_type") == "ev_ebitda",
         (col("median_multiple") >= 0.5) & (col("median_multiple") <= 60.0))
    .when(col("metric_type") == "ev_revenue",
          (col("median_multiple") >= 0.05) & (col("median_multiple") <= 25.0))
    .otherwise(lit(False))
)

display(df_flagged.filter(col("is_range_valid") == False)
        .select("sub_vertical", "metric_type", "median_multiple").limit(10))

sub_vertical,metric_type,median_multiple


### Combining rules

In [0]:
df_final = df_flagged.withColumn(
    "is_valid",
    col("is_null_valid") & col("is_positive_valid") & col("is_percentile_valid") &
    col("is_deal_count_valid") & col("is_unique_valid") & col("is_range_valid")
)

valid_records = df_final.filter(col("is_valid") == True).count()
invalid_records = total_records - valid_records
null_records = df_final.filter(col("is_null_valid") == False).count()
quality_score = round((valid_records / total_records) * 100, 2)
pipeline_status = "PASSED" if quality_score >= 95.0 else "FAILED"

print(f"Valid: {valid_records} | Invalid: {invalid_records} | Duplicates: {duplicate_count} | Nulls: {null_records}")
print(f"Quality Score: {quality_score}% | Status: {pipeline_status}")

validated_path = "abfss://silver@stmavaluationplatform.dfs.core.windows.net/silver_valuation_multiple_validated"
df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(validated_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS silver_valuation_multiple_validated
USING DELTA
LOCATION '{validated_path}'
""")

Valid: 404 | Invalid: 0 | Duplicates: 0 | Nulls: 0
Quality Score: 100.0% | Status: PASSED


DataFrame[]

### gold_data_quality_report

In [0]:
dq_report = spark.createDataFrame([{
    "run_id": run_id,
    "dataset_name": dataset_name,
    "execution_timestamp": str(datetime.datetime.now(datetime.timezone.utc)),
    "total_records": total_records,
    "valid_records": valid_records,
    "invalid_records": invalid_records,
    "duplicate_records": duplicate_count,
    "null_records": null_records,
    "quality_score": quality_score,
    "pipeline_status": pipeline_status
}])

gold_dq_path = "abfss://gold@stmavaluationplatform.dfs.core.windows.net/gold_data_quality"
dq_report.write.format("delta").mode("append").save(gold_dq_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_data_quality
USING DELTA
LOCATION '{gold_dq_path}'
""")

display(spark.sql("SELECT * FROM gold_data_quality ORDER BY execution_timestamp DESC"))

dataset_name,duplicate_records,execution_timestamp,invalid_records,null_records,pipeline_status,quality_score,run_id,total_records,valid_records
silver_valuation_multiple_staging,0,2026-08-18 05:45:23.334100+00:00,0,0,PASSED,100.0,20260818_054505,404,404
silver_valuation_multiple,0,2026-08-17 10:24:36.352369+00:00,0,0,PASSED,100.0,20260817_102416,404,404
silver_valuation_multiple,0,2026-08-17 09:20:23.168340+00:00,0,0,PASSED,100.0,20260817_085132,404,404
